
# 01A · Resumen base de general splits

Radiografía base del dashboard de equipos para la temporada 2024-25 (Regular Season). El objetivo es validar la calidad del parquet intermedio y generar tablas y visualizaciones reutilizables sobre el diferencial entre victorias y derrotas.

- Cargamos los splits consolidados y filtramos el subset `source_dataset = 0`.
- Comprobamos que cada franquicia tenga registro Win/Loss.
- Calculamos métricas relevantes, rankings y gráficos para documentar el estado base.

---



## Configuración

Importamos dependencias (`pandas`, `matplotlib`, `seaborn`) y fijamos rutas relativas partiendo del directorio del notebook. Además, creamos las carpetas donde se exportarán tablas y figuras para este reporte.

---


In [ ]:

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Configuración visual básica
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[3]
DATA_DIR = PROJECT_ROOT / "00_data"
FIGURES_DIR = PROJECT_ROOT / "00e_reports" / "figures" / "base_overview"
TABLES_DIR = PROJECT_ROOT / "00e_reports" / "tables" / "base_overview"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

INTERMEDIATE_PATH = (
    DATA_DIR
    / "00b_intermediate"
    / "team_dashboard"
    / "general_splits"
    / "2024-25"
    / "Regular Season"
    / "team_dashboard__general_splits.parquet"
)
DATASET_REFERENCE_ID = 0

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Parquet intermedio: {INTERMEDIATE_PATH}")
print(f"Dataset de referencia (source_dataset): {DATASET_REFERENCE_ID}")



## Carga de datos

Leemos el parquet consolidado (`00b_intermediate`) y filtramos exclusivamente el subset con `source_dataset = 0`, que refleja la estructura original de los splits generales. Guardamos ambos DataFrames para contrastar tamaños y columnas.

---


In [ ]:

general_df = pd.read_parquet(INTERMEDIATE_PATH)

source_dataset_col = next(
    (col for col in general_df.columns if col.lower() == "source_dataset"),
    None,
)
if source_dataset_col is None:
    raise KeyError("No se encontró la columna de referencia `source_dataset` en el parquet intermedio.")

dataset_reference_df = general_df[general_df[source_dataset_col] == DATASET_REFERENCE_ID].copy()
analysis_df = dataset_reference_df.copy()

print("general_df shape:", general_df.shape)
print("dataset_reference_df shape:", dataset_reference_df.shape)



### Vista rápida del parquet intermedio

Mostramos las primeras filas del parquet completo para reconocer nombres de columnas, tipos y valores agregados.

---


In [ ]:
display(general_df.head())



### Referencia `source_dataset = 0`

Previsualizamos el DataFrame filtrado (`analysis_df`) que usaremos en los análisis posteriores. Sirve como control de que la partición elegida preserva las métricas necesarias.

---


In [ ]:
display(analysis_df.head())



## Ingeniería de métricas derivadas

Para cubrir las estadísticas solicitadas reconstruimos proxies a partir de los conteos brutos del parquet. En cada caso se
verifica que existan las columnas necesarias antes de crear la métrica:

- $\text{OffRtg} = \frac{PTS}{MIN} \times 48$ y $\text{DefRtg} = \frac{PTS_{\text{recibidos}}}{MIN} \times 48$, con $PTS_{\text{recibidos}} = PTS - PLUS\_MINUS$.
- $\text{NetRtg} = \text{OffRtg} - \text{DefRtg}$.
- $\text{Pace} = \frac{FGA + 0.44\,FTA - OREB + TOV}{MIN} \times 48$.
- $eFG\% = \frac{FGM + 0.5\,FG3M}{FGA}$, $TOV\% = \frac{TOV}{FGA + 0.44\,FTA + TOV}$, $OREB\% = \frac{OREB}{REB}$ y $FTR = \frac{FTA}{FGA}$.

Los cálculos solo se ejecutan cuando hay datos suficientes; de lo contrario la columna queda en `NA` y el flujo continúa.

---


In [ ]:

# Cálculo seguro de métricas derivadas
COLUMN_LOOKUP = {col.upper(): col for col in analysis_df.columns}

def get_series(label):
    columna = COLUMN_LOOKUP.get(label)
    if columna is None:
        return None
    return analysis_df[columna].astype(float)

derived = []

minutos = get_series("MIN")
minutos_validos = minutos.mask(minutos == 0) if minutos is not None else None

puntos = get_series("PTS")
plus_minus = get_series("PLUS_MINUS")
if minutos_validos is not None and puntos is not None and plus_minus is not None:
    off_rating = puntos.div(minutos_validos).mul(48)
    puntos_recibidos = puntos - plus_minus
    def_rating = puntos_recibidos.div(minutos_validos).mul(48)
    analysis_df["OFF_RATING"] = off_rating
    analysis_df["DEF_RATING"] = def_rating
    analysis_df["NET_RATING"] = off_rating - def_rating
    derived.extend(["OFF_RATING", "DEF_RATING", "NET_RATING"])

fga = get_series("FGA")
fta = get_series("FTA")
oreb = get_series("OREB")
tov = get_series("TOV")
if minutos_validos is not None and fga is not None and fta is not None and oreb is not None and tov is not None:
    posesiones_estimadas = fga + 0.44 * fta - oreb + tov
    analysis_df["PACE"] = posesiones_estimadas.div(minutos_validos).mul(48)
    derived.append("PACE")

fgm = get_series("FGM")
fg3m = get_series("FG3M")
if fga is not None and fgm is not None and fg3m is not None:
    analysis_df["EFG_PCT"] = (fgm + 0.5 * fg3m).div(fga.mask(fga == 0))
    derived.append("EFG_PCT")

if fga is not None and fta is not None and tov is not None:
    tov_denom = fga + 0.44 * fta + tov
    analysis_df["TOV_PCT"] = tov.div(tov_denom.mask(tov_denom == 0))
    derived.append("TOV_PCT")

reb = get_series("REB")
if oreb is not None and reb is not None:
    analysis_df["OREB_PCT"] = oreb.div(reb.mask(reb == 0))
    derived.append("OREB_PCT")

if fta is not None and fga is not None:
    analysis_df["FTR"] = fta.div(fga.mask(fga == 0))
    derived.append("FTR")

if derived:
    print(f"✅ Métricas derivadas añadidas: {', '.join(sorted(set(derived)))}")
else:
    print("⚠️ No fue posible derivar métricas avanzadas con las columnas disponibles.")

# Actualizamos el mapeo tras crear nuevas columnas
COLUMN_LOOKUP = {col.upper(): col for col in analysis_df.columns}



## Validación Win/Loss por equipo

Verificamos que cada equipo tenga exactamente dos filas (una victoria y una derrota). Se resume cuántos resultados distintos existen por franquicia y se reportan posibles inconsistencias.

---


In [ ]:

clave_equipo = ["TEAM_ID", "TEAM_NAME"]

game_result_col = next(
    (col for col in analysis_df.columns if col.upper() == "GAME_RESULT"),
    None,
)

if game_result_col is None:
    raise KeyError(
        "No se encontró la columna `GAME_RESULT` en el subset filtrado; no es posible validar Win/Loss."
    )

wl_counts = (
    analysis_df.groupby(clave_equipo)[game_result_col]
    .agg(
        unique_result_count="nunique",
        resultados=lambda x: sorted(x.unique()),
    )
)

equipos_incompletos = wl_counts[wl_counts["unique_result_count"] != 2]
display(wl_counts.head())

if equipos_incompletos.empty:
    print("✅ Todos los equipos tienen registros de victoria y derrota.")
else:
    print("⚠️ Equipos con registros incompletos:")
    display(equipos_incompletos)



## Tablas resumen de métricas

Armamos tablas separadas para victorias y derrotas con las métricas reconstruidas (`W_PCT`, ratings, ritmo y Four
Factors), además del diferencial global \(\Delta_{\text{W-L}}\). Esto permite comparar de forma directa el cambio de promedi
os al pasar de ganar a perder.

---


In [ ]:

metricas = [
    "W_PCT",
    "NET_RATING",
    "OFF_RATING",
    "DEF_RATING",
    "PACE",
    "EFG_PCT",
    "TOV_PCT",
    "OREB_PCT",
    "FTR",
]
clave_equipo = ["TEAM_ID", "TEAM_NAME"]

COLUMN_LOOKUP = {col.upper(): col for col in analysis_df.columns}
metric_map = {metrica: COLUMN_LOOKUP.get(metrica.upper()) for metrica in metricas}
metricas_disponibles = [m for m, col in metric_map.items() if col is not None]
faltantes = sorted(set(metricas) - set(metricas_disponibles))
if faltantes:
    print(f"⚠️ Métricas ausentes en los datos filtrados: {faltantes}")

win_df = (
    analysis_df[analysis_df[game_result_col] == "W"]
    .sort_values("TEAM_NAME")
    .set_index(clave_equipo)
)
loss_df = (
    analysis_df[analysis_df[game_result_col] == "L"]
    .sort_values("TEAM_NAME")
    .set_index(clave_equipo)
)

win_metrics = win_df[[metric_map[m] for m in metricas_disponibles]].rename(
    columns={metric_map[m]: m for m in metricas_disponibles}
)
loss_metrics = loss_df[[metric_map[m] for m in metricas_disponibles]].rename(
    columns={metric_map[m]: m for m in metricas_disponibles}
)

win_table = win_metrics.rename(columns=lambda c: f"{c}_WIN")
loss_table = loss_metrics.rename(columns=lambda c: f"{c}_LOSS")
diff_table = (win_metrics - loss_metrics).rename(columns=lambda c: f"{c}_DIFF")

w_col = COLUMN_LOOKUP.get("W")
l_col = COLUMN_LOOKUP.get("L")
if w_col and l_col:
    wl_resumen = (
        analysis_df.groupby(clave_equipo)[[w_col, l_col]].max()
        .rename(columns={w_col: "W", l_col: "L"})
        .assign(W_MINUS_L=lambda df: df["W"] - df["L"])
    )
    diff_table = diff_table.join(wl_resumen[["W_MINUS_L"]], how="outer")
else:
    diff_table = diff_table.assign(W_MINUS_L=pd.NA)

win_table_reset = win_table.reset_index()
loss_table_reset = loss_table.reset_index()
diff_table_reset = diff_table.reset_index()

if metricas_disponibles:
    display(win_table_reset.head())
    display(loss_table_reset.head())
    display(diff_table_reset.head())
else:
    print(
        "ℹ️ Las tablas solo contienen identificadores porque no se encontraron métricas avanzadas disponibles."
    )

win_table_reset.to_csv(TABLES_DIR / "team_metrics_win.csv", index=False)
loss_table_reset.to_csv(TABLES_DIR / "team_metrics_loss.csv", index=False)
diff_table_reset.to_csv(TABLES_DIR / "team_metrics_diff.csv", index=False)

print("Tablas exportadas a", TABLES_DIR)



## Rankings clave

Ordenamos a las franquicias por (1) diferencial de `NET_RATING`, (2) ventaja en victorias menos derrotas y (3) prom
edios de los Four Factors derivados. Si alguna métrica falta tras el filtrado, se deja constancia para mantener tran
sparencia.

---


In [ ]:

rank_net_path = TABLES_DIR / "ranking_net_rating.csv"
rank_wl_path = TABLES_DIR / "ranking_w_minus_l.csv"
four_factor_path = TABLES_DIR / "ranking_four_factors.csv"

if "NET_RATING_DIFF" in diff_table_reset.columns and diff_table_reset["NET_RATING_DIFF"].notna().any():
    rank_net = (
        diff_table_reset
        .sort_values("NET_RATING_DIFF", ascending=False)
        .assign(
            NET_RATING_RANK=lambda df: df["NET_RATING_DIFF"].rank(
                method="dense", ascending=False
            ).astype("Int64")
        )
    )
else:
    print(
        "⚠️ No se puede generar el ranking por NET_RATING porque la métrica no está disponible tras el filtrado."
    )
    rank_net = diff_table_reset[clave_equipo].copy()
    rank_net["NET_RATING_DIFF"] = pd.NA
    rank_net["NET_RATING_RANK"] = pd.NA

if "W_MINUS_L" in diff_table_reset.columns and diff_table_reset["W_MINUS_L"].notna().any():
    rank_wl = (
        diff_table_reset
        .sort_values("W_MINUS_L", ascending=False)
        .assign(
            W_MINUS_L_RANK=lambda df: df["W_MINUS_L"].rank(
                method="dense", ascending=False
            ).astype("Int64")
        )
    )
else:
    print(
        "⚠️ No se puede generar el ranking por ΔW−L porque la columna está ausente o vacía."
    )
    rank_wl = diff_table_reset[clave_equipo].copy()
    rank_wl["W_MINUS_L"] = pd.NA
    rank_wl["W_MINUS_L_RANK"] = pd.NA

four_factor_cols = ["EFG_PCT_DIFF", "TOV_PCT_DIFF", "OREB_PCT_DIFF", "FTR_DIFF"]
orden_rangos = {
    "EFG_PCT_DIFF": False,
    "TOV_PCT_DIFF": True,
    "OREB_PCT_DIFF": False,
    "FTR_DIFF": False,
}

missing_four = [col for col in four_factor_cols if col not in diff_table_reset.columns]
if missing_four:
    print(
        f"⚠️ Columnas Four Factors ausentes en el diferencial: {missing_four}"
    )

four_factor_rank = diff_table_reset[clave_equipo].copy()
for col in four_factor_cols:
    rank_col = f"{col.replace('_DIFF', '')}_RANK"
    if col in diff_table_reset.columns and diff_table_reset[col].notna().any():
        four_factor_rank[col] = diff_table_reset[col]
        four_factor_rank[rank_col] = four_factor_rank[col].rank(
            method="dense", ascending=orden_rangos[col]
        ).astype("Int64")
    else:
        four_factor_rank[col] = pd.NA
        four_factor_rank[rank_col] = pd.NA

rank_cols = [c for c in four_factor_rank.columns if c.endswith("_RANK")]
rank_cols_validos = [c for c in rank_cols if four_factor_rank[c].notna().any()]
if rank_cols_validos:
    four_factor_rank["RANK_PROMEDIO"] = four_factor_rank[rank_cols_validos].mean(axis=1)
    four_factor_rank = four_factor_rank.sort_values("RANK_PROMEDIO")
else:
    four_factor_rank["RANK_PROMEDIO"] = pd.NA
    four_factor_rank = four_factor_rank.sort_values("TEAM_NAME")

rank_net.to_csv(rank_net_path, index=False)
rank_wl.to_csv(rank_wl_path, index=False)
four_factor_rank.to_csv(four_factor_path, index=False)

print("Rankings exportados a", TABLES_DIR)
display(rank_net.head())
display(rank_wl.head())
display(four_factor_rank.head())



## ΔNET_RATING por equipo

Gráfico de barras con el diferencial de `NET_RATING` (victorias − derrotas). Se omite si la métrica no está presente en el subset filtrado.

---


In [ ]:

if "NET_RATING_DIFF" in diff_table_reset.columns and diff_table_reset["NET_RATING_DIFF"].notna().any():
    net_diff_plot = diff_table_reset.sort_values("NET_RATING_DIFF", ascending=False)
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.barplot(
        data=net_diff_plot,
        x="NET_RATING_DIFF",
        y="TEAM_NAME",
        palette=sns.diverging_palette(240, 10, as_cmap=True),
        ax=ax,
    )
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
    ax.set_title("Diferencial de NET_RATING (W − L)")
    ax.set_xlabel("Δ NET_RATING")
    ax.set_ylabel("Equipo")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "delta_net_rating.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Figura guardada en", FIGURES_DIR / "delta_net_rating.png")
else:
    print("⚠️ No se genera la figura de ΔNET_RATING porque la métrica no está disponible.")



## Comparativa Four Factors W vs L

Promedios de los Four Factors agrupados por resultado (`W`/`L`). Si alguna métrica no está disponible, se documenta el aviso en lugar de generar la figura.

---


In [ ]:

four_factor_metricas = ["EFG_PCT", "TOV_PCT", "OREB_PCT", "FTR"]
four_factor_actual = {m: COLUMN_LOOKUP.get(m) for m in four_factor_metricas}
missing_four = [m for m, col in four_factor_actual.items() if col is None]
if missing_four:
    print(f"⚠️ Métricas Four Factors ausentes en el subset: {missing_four}")

disponibles_four = {m: col for m, col in four_factor_actual.items() if col is not None}
if disponibles_four:
    four_factor_media = (
        analysis_df.groupby(game_result_col)[list(disponibles_four.values())]
        .mean()
        .rename(columns={v: k for k, v in disponibles_four.items()})
        .reset_index()
    )
    four_factor_melt = four_factor_media.melt(
        id_vars=game_result_col,
        value_vars=list(disponibles_four.keys()),
        var_name="Métrica",
        value_name="Valor",
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=four_factor_melt, x="Métrica", y="Valor", hue=game_result_col, ax=ax)
    ax.set_title("Comparativa de Four Factors por resultado")
    ax.set_xlabel("Factor")
    ax.set_ylabel("Valor medio")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "four_factors_win_loss.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Figura guardada en", FIGURES_DIR / "four_factors_win_loss.png")
else:
    print("ℹ️ No se genera la comparativa de Four Factors porque ninguna de las métricas está disponible.")



## PACE vs OFF_RATING

Diagrama de dispersión que relaciona el ritmo (`PACE`) y la eficiencia ofensiva (`OFF_RATING`) diferenciando victorias y derrotas. Se genera únicamente cuando ambas métricas están disponibles.

---


In [ ]:

pace_col = COLUMN_LOOKUP.get("PACE")
off_rating_col = COLUMN_LOOKUP.get("OFF_RATING")
if pace_col and off_rating_col:
    scatter_df = analysis_df[[pace_col, off_rating_col, game_result_col]].rename(
        columns={pace_col: "PACE", off_rating_col: "OFF_RATING", game_result_col: "GAME_RESULT"}
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.scatterplot(
        data=scatter_df,
        x="PACE",
        y="OFF_RATING",
        hue="GAME_RESULT",
        style="GAME_RESULT",
        ax=ax,
    )
    ax.set_title("PACE vs OFF_RATING por resultado")
    ax.set_xlabel("PACE")
    ax.set_ylabel("OFF_RATING")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "pace_vs_off_rating.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Figura guardada en", FIGURES_DIR / "pace_vs_off_rating.png")
else:
    print("ℹ️ No se genera el scatter PACE vs OFF_RATING por ausencia de una o ambas métricas.")



## Distribución de NET_RATING

Histograma con KDE para comparar la distribución de `NET_RATING` según el resultado del partido. Se omite si la columna está ausente.

---


In [ ]:

net_rating_col = COLUMN_LOOKUP.get("NET_RATING")
if net_rating_col:
    net_df = analysis_df[[net_rating_col, game_result_col]].rename(
        columns={net_rating_col: "NET_RATING", game_result_col: "GAME_RESULT"}
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.histplot(
        data=net_df,
        x="NET_RATING",
        hue="GAME_RESULT",
        element="step",
        stat="density",
        common_norm=False,
        kde=True,
        ax=ax,
    )
    ax.set_title("Distribución de NET_RATING")
    ax.set_xlabel("NET_RATING")
    ax.set_ylabel("Densidad")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "net_rating_distribution.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Figura guardada en", FIGURES_DIR / "net_rating_distribution.png")
else:
    print("ℹ️ No se genera la distribución de NET_RATING porque la columna no está disponible.")



## Correlación de métricas

Heatmap de correlaciones entre las métricas avanzadas disponibles. El listado inicial incluye `W_PCT`, `NET_RATING`, `OFF_RATING`, `DEF_RATING`, `PACE`, `EFG_PCT`, `TOV_PCT`, `OREB_PCT` y `FTR`.

---


In [ ]:

corr_cols = [
    "W_PCT",
    "NET_RATING",
    "OFF_RATING",
    "DEF_RATING",
    "PACE",
    "EFG_PCT",
    "TOV_PCT",
    "OREB_PCT",
    "FTR",
]
actual_corr_cols = {col: COLUMN_LOOKUP.get(col) for col in corr_cols}
disponibles_corr = {col: real for col, real in actual_corr_cols.items() if real is not None}
missing_corr = [col for col, real in actual_corr_cols.items() if real is None]
if missing_corr:
    print(f"⚠️ Métricas ausentes para la correlación: {missing_corr}")

if len(disponibles_corr) >= 2:
    corr_matrix = (
        analysis_df[list(disponibles_corr.values())]
        .rename(columns={real: col for col, real in disponibles_corr.items()})
        .corr()
    )
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        square=True,
        ax=ax,
    )
    ax.set_title("Correlación entre métricas clave")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "metric_correlation_heatmap.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Figura guardada en", FIGURES_DIR / "metric_correlation_heatmap.png")
else:
    print("ℹ️ No se genera el heatmap de correlaciones porque hay menos de dos métricas disponibles.")
